# Low-level data reduction using ctapipe

### Check paths and environments

In [ ]:
# Where are we?
! pwd


In [ ]:
# Check for the versions of the core dependencies 
! conda list | grep ctlearn
! conda list | grep astropy
! conda list | grep ctapipe
! conda list | grep dl1-data-handler 
! conda list | grep tensorflow
! conda list | grep keras

In [ ]:
# Set here correct path to downloaded test data of CTAO simulation.
DATA_DIR = "../../../../iaa-advanced-neural-networks-2026-ctao-data"
! du -h {DATA_DIR}/simtel/*/*

In [ ]:
# Create directories to store processed data products 
!mkdir -p {DATA_DIR}/hdf5
!mkdir -p {DATA_DIR}/hdf5_merged

In [ ]:
# Exploring the ctapipe tools we are going to use in the background:
! ctapipe-process -h
! ctapipe-merge -h

### Reducing the data
The frist step is the reduction of simtel files to HDF5 files following the official CTAO data model. Several data levels can be stored within one file. As an example, we are storing here the calibrated waveforms (R1) and the integrated images (DL1a) and image parameters (DL1b). This operation is done run-wise and can be easily parallelize on a cluster using sbatch. An additional script can be found in "../scripts/run_reduction_cluster.py" for the CTAO onsite data center. The scripts used for this reduction needs to be adjusted according to the naming of the simtel file. It is recommended carefully check before a major reduction is processed.

In [ ]:
! python ../scripts/run_reduction.py \
--input_dir {DATA_DIR}/simtel/gamma-diffuse/ \
--type gamma \
--config ../configs/ctapipe_standard_LST1_config.json \
--output_dir {DATA_DIR}/hdf5/ \
--log_level INFO

We are also reducing a couple of proton files.

In [ ]:
! python ../scripts/run_reduction.py \
--input_dir {DATA_DIR}/simtel/proton/ \
--type proton \
--config ../configs/ctapipe_standard_LST1_config.json \
--output_dir {DATA_DIR}/hdf5/ \
--log_level ERROR

After production we can check the files in the output folder.

In [ ]:
! du -h {DATA_DIR}/hdf5/*

### Merging the HDF5 files
It is HIGHLY RECOMMENDED to merge the HDF5 files. Usually, the whole production is merged into 100 files, where we split 80 files into the training and validation set and reserve 20 files for the test set to build the instrumental response functions (IRFs). Working with merged files ensure a rather smooth handling of the training process with DL1DH+CTLearn.

In [ ]:
! python ../scripts/run_merger.py \
--input_dir {DATA_DIR}/hdf5/ \
--pattern "gamma*" \
--num_outputfiles 2 \
--output_dir {DATA_DIR}/hdf5_merged/ \
--log_level INFO

In [ ]:
! python ../scripts/run_merger.py \
--input_dir {DATA_DIR}/hdf5/ \
--pattern "proton*" \
--num_outputfiles 3 \
--output_dir {DATA_DIR}/hdf5_merged/ \
--log_level ERROR

In [ ]:
! du -h {DATA_DIR}/hdf5_merged/*

### Browsing through the HDF5 files via vitables
Simply install vitables ($ pip install vitables) on your machine and use this convenient GUI to explore the data model.

In [ ]:
! conda run -n vitables vitables {DATA_DIR}/hdf5_merged/*